1. Creacion de los archivos de trabajo:
    * se crean los archivos de trabajo para cada etapa del proceso de analisis, incluyendo:
        - 01_analisis_exploracion.ipynb
        - 02_preprocesamiento_de_datos.ipynb
        - 03_modelo_evolucion.ipynb
        - 04_optimizacion_de_hiperparametros.ipynb
        - 05_finalizacion_del_analisis.ipynb
    

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

def preprocess_data(df):
    """
    Función para limpiar y transformar el dataset de cáncer de piel.
    """
    df_clean = df.copy()
    
    # 1. Manejo de Valores Nulos (Basado en tu punto 2 del EDA)
    # Imputamos variables numéricas con la mediana para evitar sesgo por outliers
    num_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
        
    # Imputamos variables categóricas con la moda
    cat_cols = df_clean.select_dtypes(include=['object']).columns
    for col in cat_cols:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

    # 2. Transformación de Categorías (Encoding)
    # Label Encoding para binarias (Sexo, Tabaquismo, Antecedentes, etc.)
    le = LabelEncoder()
    bin_cols = ['Sexo', 'Tabaquismo', 'Antecedentes personales de cáncer', 
                'Inmunosupresión', 'Exposición solar crónica', 'Dolor', 
                'Ulceración o sangrado espontáneo', 'Costra persistente', 
                'Bordes elevados/irregulares', 'Endurecimiento', 'Supuración',
                'Cambio reciente de tamaño/color', 'Pérdida de peso involuntaria', 
                'Fiebre persistente', 'Fatiga']
    
    for col in bin_cols:
        if col in df_clean.columns:
            df_clean[col] = le.fit_transform(df_clean[col])

    # One-Hot Encoding para variables con múltiples categorías (Localización, Cáncer familiar)
    # Esto evita que el modelo piense que una localización es "mayor" que otra numéricamente
    multi_cat_cols = ['Cáncer familiar 1er grado (tipo)', 'Localización de la lesión', 
                      'Consumo de alcohol', 'Tiempo de evolución', 'Respuesta a antibióticos previos']
    df_clean = pd.get_dummies(df_clean, columns=multi_cat_cols, drop_first=True)

    # 3. Escalamiento de Datos (StandardScaler)
    # Vital para que la Edad (65) no domine sobre el Tamaño (0.5 cm)
    scaler = StandardScaler()
    # No escalamos el target (Antecedentes personales)
    features_to_scale = df_clean.drop('Antecedentes personales de cáncer', axis=1).columns
    df_clean[features_to_scale] = scaler.fit_transform(df_clean[features_to_scale])
    
    return df_clean